In [2]:
import numpy as np
from typing import Tuple, Optional

import jax
import jax.numpy as jnp
from jaxtyping import Array, Float, Int
import equinox as eqx

from phd.feature_search.jax_core.utils import tree_replace
from phd.feature_search.jax_core.tasks.geoff import NonlinearGEOFFTask, InputChangingGEOFFTask, BinaryRegressionTask, CoreTransientBinaryTask

In [2]:
task = InputChangingGEOFFTask(
    n_features = 10,
    n_outputs = 10,
    flip_rate = 0.0,
    n_layers = 2,
    n_stationary_layers = 0,
    hidden_dim = 20,
    weight_scale = 1.0,
    activation = "ltu",
    sparsity = 0.0,
    weight_init = "binary",
    input_bounds = [-1.0, 1.0],
    input_subspace_range = 0.5,
    input_change_freq = 1, # 40_000,
    max_input_center_change = 2.0,
    # seed = 2512161,
)

pass_fracs = []
for i in range(30):
    task, (x, y) = task.generate_batch(batch_size=100)
    y_np = np.asarray(y)
    if y_np.ndim == 1:
        y_np = y_np[:, None]
    mean = y_np.mean(axis=0)
    pct10 = np.percentile(y_np, 10, axis=0)
    pct90 = np.percentile(y_np, 90, axis=0)
    
    pass_frac = np.mean([p10 <= 0 and p90 > 0 for p10, p90 in zip(pct10, pct90)])
    pass_fracs.append(pass_frac)

    # Print all output dimensions in a single row
    # print(
    #     " ".join(
    #         f"({p10:.2f}, {p90:.2f})"
    #         for p10, p90 in zip(pct10, pct90)
    #     )
    # )
    print(
        " ".join(
            ('T' if p10 <= 0 and p90 > 0 else '_')
            for p10, p90 in zip(pct10, pct90)
        )
    )

print(f"Pass Fraction: {np.mean(pass_fracs):.3f}")


_ T _ _ T _ T T _ T
T T _ T _ T _ T _ T
_ _ _ _ T T _ T _ _
_ T _ _ T T T T _ T
_ T _ _ _ _ _ T _ T
T T _ _ T T _ _ _ T
T T _ _ T _ T T _ _
T T _ _ T T T T _ _
T T _ _ _ T _ _ _ _
T T _ _ _ T _ T _ T
T T _ T T _ _ _ T T
T T _ _ _ T T T T T
_ T _ _ _ _ _ T _ T
T _ _ _ T T T T _ T
_ _ _ _ T T _ T _ _
T T _ T _ T _ _ _ _
T _ _ _ _ _ _ _ _ _
T _ _ _ _ _ _ T _ T
_ T _ _ _ T _ T _ _
T _ _ _ T _ _ T _ _
T T _ _ _ _ _ _ _ T
_ T _ _ _ _ _ T _ _
T T _ _ _ T _ T _ _
T T _ _ T T _ _ _ T
_ _ _ _ T _ T T _ _
T T _ _ T _ _ T _ _
_ T _ _ _ T _ _ _ T
T T _ _ T T _ T _ T
T _ _ _ _ T T T _ _
T T _ _ _ T _ T _ T
Pass Fraction: 0.417


In [43]:
task = BinaryRegressionTask(
    n_features = 10,
    n_outputs = 5,
    flip_rate = 0.0,
    n_layers = 4,
    n_stationary_layers = 0,
    hidden_dim = 10,
    weight_scale = 1.0,
    sparsity = 0.0,
    input_bounds = [-1.0, 1.0],
    input_subspace_range = 0.5,
    input_change_freq = 1, # 40_000,
    max_input_center_change = 0.1,
    # seed = 2512161,
)

pass_fracs = []
output_ranges = []
for i in range(30):
    task, (x, y) = task.generate_batch(batch_size=100)
    y_np = np.asarray(y)
    if y_np.ndim == 1:
        y_np = y_np[:, None]
    mean = y_np.mean(axis=0)
    pct10 = np.percentile(y_np, 10, axis=0)
    pct90 = np.percentile(y_np, 90, axis=0)
    
    pass_frac = np.mean([p10 <= 0 and p90 > 0 for p10, p90 in zip(pct10, pct90)])
    pass_fracs.append(pass_frac)

    # Print all output dimensions in a single row
    # print(
    #     " ".join(
    #         f"({p10:.2f}, {p90:.2f})"
    #         for p10, p90 in zip(pct10, pct90)
    #     )
    # )
    # print(
    #     " ".join(
    #         ('T' if p10 <= 0 and p90 > 0 else '_')
    #         for p10, p90 in zip(pct10, pct90)
    #     )
    # )
    print(
        " ".join(str(f'{m:.2f}') for m in mean)
    )

print(f"Pass Fraction: {np.mean(pass_fracs):.3f}")


0.38 0.61 0.55 0.37 0.43
0.52 0.66 0.38 0.34 0.46
0.45 0.65 0.47 0.32 0.55
0.45 0.68 0.45 0.32 0.60
0.52 0.61 0.44 0.37 0.60
0.54 0.67 0.40 0.45 0.56
0.60 0.66 0.26 0.50 0.47
0.56 0.67 0.35 0.45 0.46
0.63 0.65 0.30 0.50 0.57
0.56 0.70 0.36 0.49 0.50
0.54 0.67 0.40 0.43 0.54
0.69 0.62 0.30 0.48 0.55
0.62 0.62 0.34 0.44 0.50
0.45 0.56 0.48 0.39 0.44
0.30 0.46 0.59 0.35 0.25
0.10 0.28 0.76 0.23 0.14
0.17 0.25 0.75 0.14 0.16
0.13 0.27 0.71 0.25 0.13
0.06 0.20 0.81 0.31 0.03
0.12 0.32 0.70 0.28 0.09
0.03 0.22 0.78 0.25 0.06
0.15 0.39 0.62 0.29 0.14
0.18 0.39 0.67 0.20 0.28
0.06 0.32 0.80 0.24 0.18
0.08 0.28 0.88 0.23 0.24
0.09 0.31 0.84 0.23 0.23
0.03 0.13 0.84 0.21 0.09
0.06 0.27 0.75 0.26 0.16
0.08 0.30 0.70 0.34 0.21
0.07 0.41 0.86 0.22 0.36
Pass Fraction: 0.913


In [34]:
task = BinaryRegressionTask(
    n_features = 10,
    n_outputs = 5,
    n_layers = 6,
    hidden_dim = 10,
    input_bounds = [-1.0, 1.0],
    input_subspace_range = 0.5,
    input_change_freq = 1, # 40_000,
    max_input_center_change = 2.0,
    seed = 2512201,
)

# First 2 layers are core representation, after that are sparesely activated
transient_hidden_dim = 2048
transient_sparsity = 0.8
transient_activation_rate = 0.01


# Change the size of and reinitialize the transient layers
rng = jax.random.PRNGKey(2512202)
for i in reversed(range(2, len(task.weights))):
    rng, key = jax.random.split(rng)
    out_dim = transient_hidden_dim if i < len(task.weights) - 1 else task.weights[-1].shape[1]
    in_dim = task.weights[1].shape[1] if i == 2 else transient_hidden_dim
    task.weights[i] = task._initialize_weights(key, in_dim, out_dim)
    task.weights[i] = task._sparsify_weights(key, task.weights[i], transient_sparsity)
    
# Steps below:
# - Change transient features so that each has a 5% chance of activation over the whole input distribution
# - For several iterations:
#   - Sample a batch with a random subspace of the input distribution
#   - Record the fraction of the time each activation in the network for each layer is active and inactive
#   - Also record the binary entropy for each activation
#   - Make a binary mask for each layer if the activation probability is < 0.025 or > 0.975 (effectively a mask based on entropy)
#   - Use the mask to easily count (sum) the number of units in each layer above the threshold given above
# - Over all iterations, print the average absolute count and fraction of units above the entropy threshold per layer
# === Part 1: Adjust transient layers for 5% activation rate ===
# We need to compute biases (thresholds) for each transient layer unit
# such that it fires ~5% of the time over the whole input distribution

# === Part 1: Adjust transient layers for 5% activation rate ===
# Compute biases layer-by-layer, using prior layer biases in each forward pass

def forward_with_activations_and_preact(task, x, biases, target_layer = None):
    """Forward pass that returns all layer activations (binary) and optionally pre-activations at target_layer."""
    activations = []
    pre_activations = None
    h = x
    for i in range(len(task.weights) - 1):  # Exclude output layer
        h = h @ task.weights[i]
        if i == target_layer:
            pre_activations = h  # Save pre-activations before applying bias/activation
        h = h - biases[i]  # Apply bias/threshold
        h = jnp.where(h > 0, 1.0, 0.0)  # LTU activation
        activations.append(h)
    return activations, pre_activations

def binary_entropy(p):
    """Compute binary entropy H(p) = -p*log(p) - (1-p)*log(1-p)."""
    p_clamped = jnp.clip(p, 1e-7, 1 - 1e-7)
    return -p_clamped * jnp.log2(p_clamped) - (1 - p_clamped) * jnp.log2(1 - p_clamped)

# ... existing code for forward_with_activations_and_preact and binary_entropy ...

# Sample from the FULL input distribution to set thresholds
rng, sample_key = jax.random.split(rng)
n_calibration_samples = 10000
full_input_samples = jax.random.uniform(
    sample_key, 
    (n_calibration_samples, task.n_features), 
    jnp.float32,
    task.input_bounds[0], 
    task.input_bounds[1],
)

# Initialize all biases to 0
n_hidden_layers = len(task.weights) - 1
transient_biases = []
for layer_idx in range(n_hidden_layers):
    if layer_idx < len(task.weights) - 1:
        layer_dim = task.weights[layer_idx].shape[1]
    else:
        layer_dim = task.weights[-1].shape[0]
    transient_biases.append(jnp.zeros(layer_dim))

# Compute biases layer-by-layer, applying previous layer biases in the forward pass
print(f"Calibrating biases for {n_hidden_layers} hidden layers...")
for layer_idx in range(n_hidden_layers):
    # Forward pass using current biases, get pre-activations at this layer
    _, pre_activations = forward_with_activations_and_preact(
        task, full_input_samples, transient_biases, target_layer = layer_idx
    )
    
    if layer_idx >= 2:  # Transient layers: set threshold for 5% activation
        # pre_activations shape: (n_samples, hidden_dim)
        # Compute percentile PER UNIT (along sample axis)
        target_percentile = 100 * (1 - transient_activation_rate)  # 95th percentile for 5% activation
        
        # Compute threshold for each unit independently
        thresholds = jnp.percentile(pre_activations, target_percentile, axis = 0)
        # thresholds shape: (hidden_dim,) - one threshold per unit
        
        transient_biases[layer_idx] = thresholds
        print(f"  Layer {layer_idx} (TRANSIENT): shape {thresholds.shape}, "
              f"thresholds min={float(jnp.min(thresholds)):.3f}, "
              f"max={float(jnp.max(thresholds)):.3f}, "
              f"mean={float(jnp.mean(thresholds)):.3f}")
    else:  # Core layers: no bias adjustment
        print(f"  Layer {layer_idx} (CORE): keeping bias = 0")

# ... rest of the code unchanged ...

# Simplified forward function for analysis (uses pre-computed biases)
def forward_with_activations(task, x, biases):
    """Forward pass that returns all layer activations (binary)."""
    activations = []
    h = x
    for i in range(len(task.weights) - 1):
        h = h @ task.weights[i]
        h = h - biases[i]
        h = jnp.where(h > 0, 1.0, 0.0)
        activations.append(h)
    return activations


# === Part 2: Collect activation statistics across iterations ===
n_iterations = 1 # 30
batch_size = 1000
entropy_threshold = 0.025  # Activation prob < 0.025 or > 0.975

# Storage for per-layer statistics
all_activation_probs = [[] for _ in range(n_hidden_layers)]
all_entropies = [[] for _ in range(n_hidden_layers)]
all_low_entropy_counts = [[] for _ in range(n_hidden_layers)]

task_iter = task  # Use a separate variable for iteration

for iteration in range(n_iterations):
    # Sample a batch with a random subspace of the input distribution
    task_iter, (x, y) = task_iter.generate_batch(batch_size = batch_size)
    
    # Get activations for all layers
    activations = forward_with_activations(task, x, transient_biases)
    
    for layer_idx, layer_activations in enumerate(activations):
        # Compute activation probability for each unit
        activation_probs = jnp.mean(layer_activations, axis = 0)
        
        # Compute binary entropy for each unit
        entropies = binary_entropy(activation_probs)
        
        # Create mask for low-entropy units (prob < 0.025 or prob > 0.975)
        low_entropy_mask = (activation_probs < entropy_threshold) | (activation_probs > 1 - entropy_threshold)
        low_entropy_count = jnp.sum(low_entropy_mask)
        
        # Store statistics
        all_activation_probs[layer_idx].append(activation_probs)
        all_entropies[layer_idx].append(entropies)
        all_low_entropy_counts[layer_idx].append(low_entropy_count)


# === Part 3: Print summary statistics ===
print("\n" + "=" * 70)
print("ACTIVATION STATISTICS SUMMARY")
print("=" * 70)
print(f"Iterations: {n_iterations}, Batch size: {batch_size}")
print(f"Entropy threshold: prob < {entropy_threshold} or prob > {1 - entropy_threshold}")
print("-" * 70)

for layer_idx in range(n_hidden_layers):
    layer_dim = transient_biases[layer_idx].shape[0]
    
    # Average counts and fractions across iterations
    counts = jnp.array(all_low_entropy_counts[layer_idx])
    mean_count = float(jnp.mean(counts))
    std_count = float(jnp.std(counts))
    mean_fraction = mean_count / layer_dim
    
    # Average activation probability across all iterations and units
    all_probs = jnp.stack(all_activation_probs[layer_idx])
    mean_activation_prob = float(jnp.mean(all_probs))
    
    # Average entropy
    all_ent = jnp.stack(all_entropies[layer_idx])
    mean_entropy = float(jnp.mean(all_ent))
    
    layer_type = "CORE" if layer_idx < 2 else "TRANSIENT"
    print(f"\nLayer {layer_idx} ({layer_type}, dim={layer_dim}):")
    print(f"  Mean activation probability: {mean_activation_prob:.4f}")
    print(f"  Mean binary entropy:         {mean_entropy:.4f}")
    print(f"  Low-entropy unit count:      {mean_count:.1f} ± {std_count:.1f}")
    print(f"  Low-entropy unit fraction:   {mean_fraction:.4f} ({mean_fraction * 100:.2f}%)")

print("\n" + "=" * 70)

Calibrating biases for 5 hidden layers...
  Layer 0 (CORE): keeping bias = 0
  Layer 1 (CORE): keeping bias = 0
  Layer 2 (TRANSIENT): shape (2048,), thresholds min=-1.000, max=5.000, mean=0.864
  Layer 3 (TRANSIENT): shape (2048,), thresholds min=0.000, max=2.000, mean=0.743
  Layer 4 (TRANSIENT): shape (2048,), thresholds min=0.000, max=15.000, mean=3.786

ACTIVATION STATISTICS SUMMARY
Iterations: 1, Batch size: 1000
Entropy threshold: prob < 0.025 or prob > 0.975
----------------------------------------------------------------------

Layer 0 (CORE, dim=10):
  Mean activation probability: 0.4430
  Mean binary entropy:         0.7210
  Low-entropy unit count:      1.0 ± 0.0
  Low-entropy unit fraction:   0.1000 (10.00%)

Layer 1 (CORE, dim=10):
  Mean activation probability: 0.4076
  Mean binary entropy:         0.6313
  Low-entropy unit count:      2.0 ± 0.0
  Low-entropy unit fraction:   0.2000 (20.00%)

Layer 2 (TRANSIENT, dim=2048):
  Mean activation probability: 0.0000
  Mean bin

In [7]:
# Create the task with similar configuration to before
task = CoreTransientBinaryTask(
    n_features = 10,
    n_core_layers = 2,
    core_hidden_dim = 20,
    core_sparsity = 0.0,
    n_transient_layers = 2,
    transient_hidden_dim = 2048,
    transient_sparsity = 0.0,
    transient_activation_rate = 0.01,
    n_outputs = 5,
    weight_scale = 1.0,
    input_bounds = (-1.0, 1.0),
    input_subspace_range = 0.5,
    input_change_freq = 1,
    max_input_center_change = 2.0,
    n_calibration_samples = 10000,
    seed = 2512201,
)

print(f"Task created with {task.n_layers} total layers:")
print(f"  Core layers: {task.n_core_layers} (dim={task.core_hidden_dim})")
print(f"  Transient layers: {task.n_layers - task.n_core_layers - 1} (dim={task.transient_hidden_dim})")
print(f"  Target transient activation rate: {task.transient_activation_rate}")
print()

# === Helper functions ===
def binary_entropy(p):
    """Compute binary entropy H(p) = -p*log(p) - (1-p)*log(1-p)."""
    p_clamped = jnp.clip(p, 1e-7, 1 - 1e-7)
    return -p_clamped * jnp.log2(p_clamped) - (1 - p_clamped) * jnp.log2(1 - p_clamped)

def forward_with_activations(task, x):
    """Forward pass that returns all hidden layer activations (binary)."""
    activations = []
    h = x
    for i in range(task.n_layers - 1):  # Exclude output layer
        h = h @ task.weights[i]
        h = h - task.layer_biases[i]
        h = jnp.where(h > 0, 1.0, 0.0)
        activations.append(h)
    return activations


# === Collect activation statistics across iterations ===
n_iterations = 30
batch_size = 1000
entropy_threshold = 0.025  # Activation prob < 0.025 or > 0.975

n_hidden_layers = task.n_layers - 1

# Storage for per-layer statistics
all_activation_probs = [[] for _ in range(n_hidden_layers)]
all_entropies = [[] for _ in range(n_hidden_layers)]
all_low_entropy_counts = [[] for _ in range(n_hidden_layers)]

task_iter = task

for iteration in range(n_iterations):
    # Sample a batch with a random subspace of the input distribution
    task_iter, (x, y) = task_iter.generate_batch(batch_size = batch_size)
    
    # Get activations for all hidden layers
    activations = forward_with_activations(task, x)
    
    for layer_idx, layer_activations in enumerate(activations):
        # Compute activation probability for each unit
        activation_probs = jnp.mean(layer_activations, axis = 0)
        
        # Compute binary entropy for each unit
        entropies = binary_entropy(activation_probs)
        
        # Create mask for low-entropy units (prob < 0.025 or prob > 0.975)
        low_entropy_mask = (activation_probs < entropy_threshold) | (activation_probs > 1 - entropy_threshold)
        low_entropy_count = jnp.sum(low_entropy_mask)
        
        # Store statistics
        all_activation_probs[layer_idx].append(activation_probs)
        all_entropies[layer_idx].append(entropies)
        all_low_entropy_counts[layer_idx].append(low_entropy_count)


# === Print summary statistics ===
print("=" * 70)
print("ACTIVATION STATISTICS SUMMARY")
print("=" * 70)
print(f"Iterations: {n_iterations}, Batch size: {batch_size}")
print(f"Low-entropy threshold: prob < {entropy_threshold} or prob > {1 - entropy_threshold}")
print("-" * 70)

for layer_idx in range(n_hidden_layers):
    layer_dim = task.layer_biases[layer_idx].shape[0]
    is_core = layer_idx < task.n_core_layers
    layer_type = "CORE" if is_core else "TRANSIENT"
    expected_rate = 0.5 if is_core else task.transient_activation_rate
    
    # Average counts and fractions across iterations
    counts = jnp.array(all_low_entropy_counts[layer_idx])
    mean_count = float(jnp.mean(counts))
    std_count = float(jnp.std(counts))
    mean_fraction = mean_count / layer_dim
    
    # Average activation probability across all iterations and units
    all_probs = jnp.stack(all_activation_probs[layer_idx])
    mean_activation_prob = float(jnp.mean(all_probs))
    
    # Average entropy
    all_ent = jnp.stack(all_entropies[layer_idx])
    mean_entropy = float(jnp.mean(all_ent))
    
    # Check if activation rate is close to expected
    rate_error = abs(mean_activation_prob - expected_rate)
    rate_ok = "✓" if rate_error < 0.1 else "✗"
    
    print(f"\nLayer {layer_idx} ({layer_type}, dim={layer_dim}):")
    print(f"  Expected activation rate:    {expected_rate:.4f}")
    print(f"  Mean activation probability: {mean_activation_prob:.4f} {rate_ok}")
    print(f"  Mean binary entropy:         {mean_entropy:.4f}")
    print(f"  Low-entropy unit count:      {mean_count:.1f} ± {std_count:.1f}")
    print(f"  Low-entropy unit fraction:   {mean_fraction:.4f} ({mean_fraction * 100:.2f}%)")

print("\n" + "=" * 70)
print("EXPECTED BEHAVIOR:")
print("  - Core layers should have ~50% activation rate (bias=0)")
print("  - Transient layers should have ~5% activation rate (calibrated)")
print("  - Transient layers should have more low-entropy units")
print("    (since many units rarely/never fire in a given subspace)")
print("=" * 70)

Task created with 5 total layers:
  Core layers: 2 (dim=20)
  Transient layers: 2 (dim=2048)
  Target transient activation rate: 0.01

ACTIVATION STATISTICS SUMMARY
Iterations: 30, Batch size: 1000
Low-entropy threshold: prob < 0.025 or prob > 0.975
----------------------------------------------------------------------

Layer 0 (CORE, dim=20):
  Expected activation rate:    0.5000
  Mean activation probability: 0.4946 ✓
  Mean binary entropy:         0.6898
  Low-entropy unit count:      1.8 ± 2.5
  Low-entropy unit fraction:   0.0900 (9.00%)

Layer 1 (CORE, dim=20):
  Expected activation rate:    0.5000
  Mean activation probability: 0.4824 ✓
  Mean binary entropy:         0.4585
  Low-entropy unit count:      6.1 ± 2.8
  Low-entropy unit fraction:   0.3033 (30.33%)

Layer 2 (TRANSIENT, dim=2048):
  Expected activation rate:    0.0100
  Mean activation probability: 0.0040 ✓
  Mean binary entropy:         0.0258
  Low-entropy unit count:      1961.4 ± 62.0
  Low-entropy unit fraction: 